# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения

Выполнили студенты гр. 2384 Федоров Михаил и Муравин Егор. Вариант №30

## Цель работы

Получение практических навыков нахождения точечных статистических оценок параметров распределения.

## Постановка задачи

Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.

## Основные теоретические положения

### 1. Выборочное среднее

Статистическая оценка математического ожидания определяется по формуле:

$$
\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i.
$$

Для интервального ряда:

$$
\bar{x} = \frac{1}{n} \sum n_i x_i,
$$

где $ x_i $ — середина интервала, $ n_i $ — частота интервала.


### 2. Выборочная дисперсия

Смещённая оценка дисперсии:

$$
D_в = \frac{1}{n} \sum (x_i - \bar{x})^2.
$$

Несмещённая (исправленная) оценка дисперсии:

$$
s^2 = \frac{1}{n-1} \sum (x_i - \bar{x})^2.
$$

Среднеквадратичное отклонение:

$$
\sigma_в = \sqrt{D_в}, \qquad s = \sqrt{s^2}.
$$


### 3. Метод условных вариант

Для упрощения вычислений в интервальных рядах используют условные варианты:

$$
u_j = \frac{x_j - C}{h},
$$

где  
$ x_j $ — середина интервала,  
$ C $ — ложный нуль (условное начало отсчёта),  
$ h $ — ширина интервала.

Условный момент k-го порядка:

$$
\bar{M}_k^* = \frac{1}{N}\sum n_j u_j^k.
$$

Через условные моменты вычисляются центральные моменты:

$$
m_2 = (M_2^* - (M_1^*)^2) h^2,
$$

$$
m_3 = (M_3^* - 3M_2^*M_1^* + 2(M_1^*)^3) h^3,
$$

$$
m_4 = (M_4^* - 4M_3^*M_1^* + 6M_2^*(M_1^*)^2 - 3(M_1^*)^4) h^4.
$$


### 4. Коэффициенты асимметрии и эксцесса

Коэффициент асимметрии:

$$
A_s = \frac{m_3}{\sigma^3}.
$$

Если $ A_s > 0 $ — распределение правосторонне асимметрично,  
если $ A_s < 0 $ — левосторонне асимметрично.

Коэффициент эксцесса:

$$
E = \frac{m_4}{\sigma^4} - 3.
$$

Если $ E > 0 $ — распределение более островершинное по сравнению с нормальным,  
если $ E < 0 $ — более плосковершинное.


### 5. Мода и медиана для интервального ряда

Мода вычисляется по формуле:

$$
M_o = x_m + h \cdot 
\frac{f_m - f_{m-1}}
{(f_m - f_{m-1}) + (f_m - f_{m+1})},
$$

где  
$ x_m $ — нижняя граница модального интервала,  
$ f_m $ — частота модального интервала.

Медиана:

$$
M_e = x_{me} + h \cdot 
\frac{\frac{n}{2} - S_{me-1}}
{f_{me}},
$$

где  
$ x_{me} $ — нижняя граница медианного интервала,  
$ S_{me-1} $ — накопленная частота до медианного интервала.


### 6. Коэффициент вариации

$$
V = \frac{\sigma}{\bar{x}} \cdot 100\%.
$$

Если $ V < 33\% $, совокупность считается однородной; при больших значениях вариация признака считается значительной.


## Выполнение работы


### 1. Для середин интервального ряда, полученного в практической работе №1, вычислить условные варианты. Заполнить табл. 1 (в последней строке Σ необходимо заполнить суммы столбцов; ячейки отмеченные прочерком заполнять не надо). Провести контроль вычислений.

In [24]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import math

In [25]:
df = pd.read_csv("data.csv", sep=",")
df.rename(columns={"nu": "relative_weight", "E": "simplicity"}, inplace=True)
df = df.sample(n=118, random_state=1)
df

,relative_weight,simplicity
94,458,133.5
54,440,126.7
59,437,129.2
115,429,112.9
74,503,149.9
...,...,...
9,566,175.7
72,436,114.3
12,500,155.5
107,471,119.7


In [26]:
# Интервальный ряд
def build_intervals_row(data):
    x_min = data.min()
    x_max = data.max()
    n = len(data)
    k = math.ceil(1 + 3.322 * math.log10(n))
    print(f"Количество интервалов: {k}")
    print(f"x_min = {x_min}, x_max = {x_max}")
    h = (x_max - x_min) / k
    print(f"Длина шага: {h:.2f}")


    boundaries = [x_min]
    for _ in range(k):
        boundaries.append(round(boundaries[-1] + h, 2))
    
    if boundaries[-1] < x_max:
        boundaries.append(round(boundaries[-1] + h, 2))
    
    print(f"Фактически интервалов: {len(boundaries) - 1}")

    counts, edges = np.histogram(data, bins=boundaries)
    midpoints = (edges[:-1] + edges[1:]) / 2
    midpoints_rounded = [round(m, 2) for m in midpoints]
    
    
    return midpoints_rounded, counts, h, n, edges, midpoints


In [27]:
def solve_table_one(C, h, midpoints, counts):
    u = (midpoints - C) / h
    n_u = counts * u
    n_u2 = n_u * u
    n_u3 = n_u2 * u
    n_u4 = n_u3 * u
    n_u1_4 = counts * ((u + 1) ** 4)


    table1_data = {
        "x_i": list(midpoints) + ["-"],
        "n_i": list(counts) + [sum(counts)],
        "u_i": list(u) + ["-"],
        "n_i * u_i": list(n_u) + [round(np.sum(n_u), 4)],
        "n_i * u_i^2": list(n_u2) + [round(np.sum(n_u2), 4)],
        "n_i * u_i^3": list(n_u3) + [round(np.sum(n_u3), 4)],
        "n_i * u_i^4": list(n_u4) + [round(np.sum(n_u4), 4)],
        "n_i * (u_i + 1)^4": list(map(lambda x: round(x, 4), n_u1_4)) + [round(np.sum(n_u1_4), 4)]
    }

    # Контроль вычислений
    left = round(np.sum(n_u1_4), 4)
    right = round(np.sum(n_u4), 4) + 4*round(np.sum(n_u3), 4) + 6*round(np.sum(n_u2), 4) + 4*round(np.sum(n_u), 4) + sum(counts)

    print(f"Контроль вычислений: {left=}, {right=}, {right == left}")

    df_table1_rw = pd.DataFrame(table1_data)
    return df_table1_rw, n_u, n_u2, n_u3, n_u4, n_u1_4


In [28]:
# relative_weight
print(f"Признак: relative_weight")
midpoints_rw, counts_rw, h_rw, n_rw, edges_rw, midpoints_exact_rw = build_intervals_row(df["relative_weight"])
print(f"Середины интервалов: {midpoints_rw}")
print(f"Частоты: {counts_rw}")

Признак: relative_weight
Количество интервалов: 8
x_min = 320, x_max = 623
Длина шага: 37.88
Фактически интервалов: 8
Середины интервалов: [np.float64(338.94), np.float64(376.82), np.float64(414.7), np.float64(452.58), np.float64(490.46), np.float64(528.34), np.float64(566.22), np.float64(604.1)]
Частоты: [ 2 10 27 38 20 15  4  2]


In [29]:
# В качестве ложного нуля выберем середину 3 интервала.

# Ложный нуль
C_rel = midpoints_exact_rw[3] 
# Длина шага
h_rel = h_rw
table_rel, n_u_rel, n_u2_rel, n_u3_rel, n_u4_rel, n_u1_4_rel = solve_table_one(C_rel, h_rel, midpoints_exact_rw, counts_rw)
table_rel

Контроль вычислений: left=np.float64(3890.4819), right=np.float64(3890.4818), False


,x_i,n_i,u_i,n_i * u_i,n_i * u_i^2,n_i * u_i^3,n_i * u_i^4,n_i * (u_i + 1)^4
0,338.94,2,-3.000396,-6.000792,18.004753,-54.021389,162.085561,32.0254
1,376.82,10,-2.000264,-20.002640,40.010562,-80.031687,160.084505,10.0106
2,414.7,27,-1.000132,-27.003564,27.007129,-27.010694,27.014260,0.0000
3,452.58,38,0.0,0.000000,0.000000,0.000000,0.000000,38.0000
4,490.46,20,1.000132,20.002640,20.005281,20.007922,20.010563,320.0845
5,528.34,15,2.000264,30.003960,60.015843,120.047531,240.126758,1215.4278
6,566.22,4,3.000396,12.001584,36.009506,108.042778,324.171123,1024.4056
7,604.1,2,4.000528,8.001056,32.008449,128.050700,512.270417,1250.5281
8,-,118,-,17.002200,233.061500,215.085200,1445.763200,3890.4819


In [30]:
# simplicity
print(f"Признак: simplicity")
midpoints_s, counts_s, h_s, n_s, edges_s, midpoints_exact_s = build_intervals_row(df["simplicity"])
print(f"Середины интервалов (x_i): {midpoints_s}")
print(f"Частоты (n_i): {counts_s}")

Признак: simplicity
Количество интервалов: 8
x_min = 71.1, x_max = 195.7
Длина шага: 15.57
Фактически интервалов: 8
Середины интервалов (x_i): [np.float64(78.89), np.float64(94.47), np.float64(110.05), np.float64(125.63), np.float64(141.2), np.float64(156.78), np.float64(172.35), np.float64(187.92)]
Частоты (n_i): [ 4  6 24 38 24 14  5  3]


In [31]:
# В качестве ложного нуля выберем середину 3 интервала.

# Ложный нуль
C_simple = midpoints_exact_s[3] 
# Длина шага
h_simple = h_s
table_simple, n_u_simple, n_u2_simple, n_u3_simple, n_u4_simple, n_u1_4_simple = solve_table_one(C_simple, h_simple, midpoints_exact_s, counts_s)
table_simple

Контроль вычислений: left=np.float64(4779.7611), right=np.float64(4779.7608), False


,x_i,n_i,u_i,n_i * u_i,n_i * u_i^2,n_i * u_i^3,n_i * u_i^4,n_i * (u_i + 1)^4
0,78.89,4,-3.000963,-12.003852,36.023118,-108.104046,324.416252,64.1234
1,94.47,6,-2.000642,-12.003852,24.015412,-48.046243,96.123334,6.0154
2,110.05,24,-1.000321,-24.007705,24.015412,-24.023121,24.030833,0.0000
3,125.63,38,0.0,0.000000,0.000000,0.000000,0.000000,38.0000
4,141.205,24,1.0,24.000000,24.000000,24.000000,24.000000,384.0000
5,156.775,14,1.999679,27.995506,55.982024,111.946076,223.856214,1133.5147
6,172.35,5,2.999679,14.998395,44.990370,134.956666,404.826673,1279.5891
7,187.925,3,3.999679,11.999037,47.992296,191.953776,767.753481,1874.5185
8,-,118,-,30.977500,257.018600,282.683100,1865.006800,4779.7611


### 2. Вычислить условные эмпирические моменты  $\bar{M}^{\ast}_{k}$ через условные варианты. С помощью условных эмпирических моментов вычислить центральные эмпирические моменты $\bar{m}_{k}$. Полученные результаты занести в табл. 2.

In [32]:
def compute_moments(n_u, n_u2, n_u3, n_u4, n, h):
    M1_star = np.sum(n_u) / n
    M2_star = np.sum(n_u2) / n
    M3_star = np.sum(n_u3) / n
    M4_star = np.sum(n_u4) / n

    
    m1 = 0
    m2 = (M2_star - M1_star**2) * (h**2)
    m3 = (M3_star - 3*M2_star*M1_star + 2*M1_star**3) * (h**3)
    m4 = (M4_star - 4*M3_star*M1_star + 6*M2_star*M1_star**2 - 3*M1_star**4) * (h**4)

    table2_data = {
        "k": [1, 2, 3, 4],
        "M*_k": [round(M1_star, 4), round(M2_star, 4), round(M3_star, 4), round(M4_star, 4)],
        "m_k": [round(m1, 4), round(m2, 4), round(m3, 4), round(m4, 4)]
    }
    df_table2_rw = pd.DataFrame(table2_data)
    return df_table2_rw, m1, m2, m3, m4


In [33]:
# relative_weight
print(f"Признак: relative_weight")
moments_real, m1_rel, m2_rel, m3_rel, m4_rel = compute_moments(n_u_rel, n_u2_rel, n_u3_rel, n_u4_rel, n_rw, h_rw)
moments_real

Признак: relative_weight


,k,M*_k,m_k
0,1,0.1441,0.000000e+00
1,2,1.9751,2.803526e+03
2,3,1.8228,5.297298e+04
3,4,12.2522,2.355486e+07


In [34]:
# simplicity
print(f"Признак: simplicity")
moments_simple, m1_simple, m2_simple, m3_simple, m4_simple = compute_moments(n_u_simple, n_u2_simple, n_u3_simple, n_u4_simple, n_rw, h_simple)
moments_simple

Признак: simplicity


,k,M*_k,m_k
0,1,0.2625,0.0000
1,2,2.1781,511.6526
2,3,2.3956,2706.6657
3,4,15.8051,834189.0350


### 3. Вычислить выборочные среднее $\bar{x}_в$ и дисперсию $D_в$ с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Вычислить выборочное СКО $\sigma_в$.

In [35]:
def solve_three(C, counts, M1_star, h, midpoints):
    # Стандартный способ
    x_in = np.average(midpoints, weights=counts)
    print(f"Стандартный способ: x_в = {x_in:.4f}")

    # Способ условных вариант
    x_in_cond = C + M1_star * h
    print(f"Способ условных вариант: x_в = {x_in_cond:.4f}")
    print(f"Результаты совпадают: {abs(x_in - x_in_cond) < 1e-2}")
    
    
    # Выборочная дисперсия
    D_in = np.average((midpoints - x_in)**2, weights=counts)
    print(f"\nВыборочная дисперсия стандарт D_в = {D_in:.4f}")
    D_in_cond = np.average((midpoints - x_in_cond)**2, weights=counts)
    print(f"Выборочная дисперсия условные варианты D_в = {D_in_cond:.4f}")
    print(f"Результаты совпадают: {abs(D_in - D_in_cond) < 1e-2}")
    


    # Выборочное СКО 
    sigma_in = np.sqrt(D_in)
    print(f"Выборочное СКО = {sigma_in:.4f}")
    
    return D_in, sigma_in, x_in

In [36]:
# simplicity
print(f"Признак: simplicity")

D_in_simple, sigma_in_simple, x_in_simple = solve_three(C_simple, counts_s, np.sum(n_u_simple) / n_s, h_s, midpoints_s)


Признак: simplicity
Стандартный способ: x_в = 129.7182
Способ условных вариант: x_в = 129.7188
Результаты совпадают: True

Выборочная дисперсия стандарт D_в = 511.6466
Выборочная дисперсия условные варианты D_в = 511.6466
Результаты совпадают: True
Выборочное СКО = 22.6196


In [37]:
# relative_weight
print(f"Признак: relative_weight")

D_in_rel, sigma_in_rel, x_in_rel = solve_three(C_rel, counts_rw, np.sum(n_u_rel) / n_rw, h_rel, midpoints_rw)

Признак: relative_weight
Стандартный способ: x_в = 458.0373
Способ условных вариант: x_в = 458.0373
Результаты совпадают: True

Выборочная дисперсия стандарт D_в = 2803.5264
Выборочная дисперсия условные варианты D_в = 2803.5264
Результаты совпадают: True
Выборочное СКО = 52.9483


### 4. Вычислить исправленную выборочную дисперсию $s^2$ и исправленное СКО s. Сравнить данные оценки с смещёнными оценками дисперсии и СКО. Сделать выводы.

In [38]:
def solve_four(D_in, n, sigma_in):
    # Исправленная выборочная дисперсия
    s2 = D_in * n / (n - 1)
    print(f"Исправленная дисперсия s^2 = {s2:.4f}")

    # Исправленное СКО
    s = np.sqrt(s2)
    print(f"Исправленное СКО s = {s:.4f}")

    print(f"Сравнение:")
    print(f"D_в (смещенная) = {D_in:.4f} | s^2 (несмещенная) = {s2:.4f}")
    print(f"σ_в (смещенное) = {sigma_in:.4f} | s (несмещенное) = {s:.4f}")

In [39]:
# relative_weight
print(f"Признак: relative_weight")

solve_four(D_in_rel, n_rw, sigma_in_rel)

Признак: relative_weight
Исправленная дисперсия s^2 = 2827.4882
Исправленное СКО s = 53.1741
Сравнение:
D_в (смещенная) = 2803.5264 | s^2 (несмещенная) = 2827.4882
σ_в (смещенное) = 52.9483 | s (несмещенное) = 53.1741


In [40]:
# simplicity
print(f"Признак: simplicity")

solve_four(D_in_simple, n_s, sigma_in_simple)

Признак: simplicity
Исправленная дисперсия s^2 = 516.0196
Исправленное СКО s = 22.7161
Сравнение:
D_в (смещенная) = 511.6466 | s^2 (несмещенная) = 516.0196
σ_в (смещенное) = 22.6196 | s (несмещенное) = 22.7161


##### Вывод
Исправленные оценки немного больше смещенных. Это связано со способом расчета, который подразумевает умножение смещенной дисперсии на n / (n - 1), что гарантировано будет больше 1. Можно заметить, что при увеличении объема выборки различие в дисперсиях стремится к нулю

### 5. Найти статистическую оценку коэффициентов асимметрии $\bar{A}_s$ и эксцесса $\bar{E}$. Сделать выводы.

In [41]:
def solve_five(m3, m4, sigma_in):
    # Асимметрия 
    A = m3 / (sigma_in**3)
    print(f"Коэффициент асимметрии A = {A:.4f}")

    # Эксцесс
    E = m4 / (sigma_in**4) - 3
    print(f"Коэффициент эксцесса E = {E:.4f}")


In [42]:
# relative_weight
print(f"Признак: relative_weight")

solve_five(m3_rel, m4_rel, sigma_in_rel)

Признак: relative_weight
Коэффициент асимметрии A = 0.3569
Коэффициент эксцесса E = -0.0031


##### Вывод
Положительная асимметрия указывает на правостороннюю асимметрию. Хвост распределения длиннее справа, большинство значений сосредоточены слева от пика.
Отрицательный эксцесс означает, что выбросы в данных менее интенсивны, чем для нормального распределения.

In [43]:
# simplicity
print(f"Признак: simplicity")

solve_five(m3_simple, m4_simple, sigma_in_simple)

Признак: simplicity
Коэффициент асимметрии A = 0.2339
Коэффициент эксцесса E = 0.1866


##### Вывод
Положительная асимметрия указывает на правостороннюю асимметрию. Хвост распределения длиннее справа, большинство значений сосредоточены слева от пика.
Положительный эксцесс означает, что распределение имеет более острую вершину. В данном случае эксцесс близок к нулю, что говорит о близости формы распределения к нормальному.

### 6. Для интервального ряда вычислить моду $M_o^*$, медиану $M_e^*$ и коэффициент вариации $V^*$ заданного распределения. Сделать выводы.

In [44]:
def solve_six(counts, edges, h, n, x_in, sigma_in):
    # Индекс интервала с максимальной частотой
    mo_interval_index = np.argmax(counts)
    
    # Частота модального интервала
    f_mo = counts[mo_interval_index]
    
    f_mo_minus_1 = counts[mo_interval_index - 1] if mo_interval_index > 0 else 0
    
    f_mo_plus_1 = counts[mo_interval_index + 1] if mo_interval_index < len(counts)-1 else 0
    
    # Левая граница модального интервала
    x_mo = edges[mo_interval_index]
    
    # Мода
    if (f_mo - f_mo_minus_1) + (f_mo - f_mo_plus_1) != 0:
        Mo = x_mo + h * (f_mo - f_mo_minus_1) / ((f_mo - f_mo_minus_1) + (f_mo - f_mo_plus_1))
    else:
        Mo = x_mo
    
    # Расчет медианы
    n_2 = n / 2
    
    cumulative = 0
    me_interval_index = 0
    for i, count in enumerate(counts):
        cumulative += count
        if cumulative >= n_2:
            me_interval_index = i
            break
    
    # Накопленная частота до медианного интервала
    S_me_minus_1 = cumulative - counts[me_interval_index]
    
    # Левая граница медианного интервала
    x_me = edges[me_interval_index]
    
    # Частота медианного интервала
    f_me = counts[me_interval_index]
    
    # Медиана
    Me = x_me + h * (n_2 - S_me_minus_1) / f_me
    
    # Расчет коэф вариации
    V = (sigma_in / x_in) * 100
    
    print(f"Мода M_o = {Mo:.4f}")
    print(f"Медиана M_e = {Me:.4f}")
    print(f"Коэффициент вариации V = {V:.2f}%")


In [45]:
# relative_weight
print(f"Признак: relative_weight")

solve_six(counts_rw, edges_rw, h_rel, n_rw, x_in_rel, sigma_in_rel)

Признак: relative_weight
Мода M_o = 448.0064
Медиана M_e = 453.5742
Коэффициент вариации V = 11.56%


##### Вывод
Мода меньше медианы, что говорит о наличии правосторонней асимметрии в распределении. Коэффициент вариации составляет 11.56%, что ниже уровня в 33%, следовательно, совокупность данных по данному признаку является качественно однородной, а разброс значений вокруг центра распределения оценивается как слабый.

In [46]:
# simplicity
print(f"Признак: simplicity")

solve_six(counts_s, edges_s, h_simple, n_s, x_in_simple, sigma_in_simple)


Признак: simplicity
Мода M_o = 125.6275
Медиана M_e = 128.0867
Коэффициент вариации V = 17.44%


##### Вывод
Результат аналогичен предыдущему, однако значение коэффициента вариации выше, чем у первого признака, что указывает на более выраженную изменчивость признака simplicity. Его значения варьируются относительно среднего уровня сильнее, чем значения relative_weight.

### **Выводы**
В ходе работы для выборочных данных были найдены точечные оценки основных параметров распределения двумя способами: методом моментов и методом условных вариант. Полученные значения выборочного среднего, дисперсии и среднеквадратичного отклонения, вычисленные разными методами, совпали, что подтверждает корректность расчетов. Исправленные оценки дисперсии и СКО оказались незначительно больше смещённых, причём разница уменьшается с ростом объёма выборки.

Анализ коэффициентов асимметрии и эксцесса показал наличие слабой правосторонней асимметрии распределений и формы, близкой к нормальной. Значения коэффициента вариации для обоих признаков не превышают 33%, что свидетельствует об однородности совокупности и умеренной изменчивости признаков. В целом распределения можно считать близкими к нормальному закону.